# Exercise 4: Evaluation

In this exercise, you will evaluate your fine-tuned LLM using perplexity and qualitative assessment. You'll compare experiments in MLflow.

## Learning Objectives

By the end of this exercise, you will be able to:
- Calculate perplexity for language models
- Perform qualitative evaluation of model outputs
- Compare experiments in MLflow
- Understand LLM evaluation challenges

## Prerequisites

Before starting this exercise, ensure you have:
1. Completed Exercise 3: LoRA Tuning
2. A fine-tuned LoRA adapter
3. MLflow tracking set up

## Step 1: Import Required Libraries

Let's start by importing the necessary libraries for evaluation:

In [ ]:
# Import necessary libraries
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import mlflow
import numpy as np
import json
import os

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")

## Step 2: Load Base Model and Fine-tuned Adapter

Now we'll load both the base model and our fine-tuned LoRA adapter for comparison.

In [ ]:
# Model configuration
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "./lora_adapter"  # Path to your saved LoRA adapter

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load fine-tuned model with LoRA adapter
print("Loading fine-tuned model with LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Models loaded successfully!")

## Step 3: Calculate Perplexity

Perplexity measures how well the model predicts a sequence. Lower perplexity indicates better performance.

We'll implement a function to calculate perplexity for a given text.

In [ ]:
# Function to calculate perplexity
def calculate_perplexity(text, model, tokenizer):
    """Calculate perplexity for a given text."""
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids
    
    # Move to same device as model
    input_ids = input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
        perplexity = math.exp(loss.item())
    
    return perplexity

# Test on example texts
test_texts = [
    "MLOps is the practice of deploying and maintaining machine learning models in production reliably and efficiently.",
    "The key components of an MLOps pipeline include data collection, preparation, training, evaluation, deployment, and monitoring.",
    "Model monitoring helps track performance over time and detect when models need retraining due to data drift."
]

print("Perplexity Evaluation:")
print("=" * 50)
for i, text in enumerate(test_texts):
    base_ppl = calculate_perplexity(text, base_model, tokenizer)
    fine_tuned_ppl = calculate_perplexity(text, model, tokenizer)
    print(f"Text {i+1}:")
    print(f"  Base Model Perplexity: {base_ppl:.2f}")
    print(f"  Fine-tuned Model Perplexity: {fine_tuned_ppl:.2f}")
    print(f"  Improvement: {base_ppl - fine_tuned_ppl:.2f}")
    print()

## Step 4: Qualitative Evaluation

Now let's compare the outputs of the base model and fine-tuned model on the same prompts.
We'll create a function to generate responses and then compare them qualitatively.

In [ ]:
# Function to generate response
def generate_response(prompt, model, tokenizer, max_new_tokens=100, temperature=0.7):
    """Generate response from model given a prompt."""
    # Format prompt for instruction tuning
    formatted_prompt = f"""### Instruction:
{prompt}

### Input:

### Response:
"""
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part (after "### Response:")
    if "### Response:" in full_response:
        response = full_response.split("### Response:")[1].strip()
    else:
        response = full_response
    
    return response

# Test prompts
test_prompts = [
    "What is MLOps and why is it important?",
    "Explain the difference between traditional DevOps and MLOps.",
    "What are the key challenges in deploying machine learning models to production?",
    "How does model monitoring help in maintaining ML systems over time?"
]

print("Qualitative Evaluation:")
print("=" * 50)
for i, prompt in enumerate(test_prompts):
    print(f"\nPrompt {i+1}: {prompt}")
    print("-" * 50)
    
    base_response = generate_response(prompt, base_model, tokenizer)
    fine_tuned_response = generate_response(prompt, model, tokenizer)
    
    print("Base Model Response:")
    print(base_response)
    print()
    print("Fine-tuned Model Response:")
    print(fine_tuned_response)
    print()
    print("=" * 50)

## Step 5: Compare Experiments in MLflow

Now let's explore how to compare different experiments in MLflow to track our progress.

In [ ]:
# Connect to MLflow and list experiments
print("MLflow Experiments:")
print("=" * 30)

experiments = mlflow.list_experiments()
for exp in experiments:
    print(f"Experiment ID: {exp.experiment_id}")
    print(f"Name: {exp.name}")
    print(f"Artifact Location: {exp.artifact_location}")
    print()

# Get runs from the LoRA tuning experiment
lora_exp_id = None
for exp in experiments:
    if "lora" in exp.name.lower() or "tuning" in exp.name.lower():
        lora_exp_id = exp.experiment_id
        break

if lora_exp_id:
    print(f"\nRuns in LoRA Tuning Experiment (ID: {lora_exp_id}):")
    print("=" * 50)
    
    runs = mlflow.search_runs(experiment_ids=[lora_exp_id])
    if len(runs) > 0:
        # Show recent runs
        for idx, run in runs.head().iterrows():
            print(f"Run ID: {run['run_id']}")
            print(f"  Status: {run['status']}")
            print(f"  Start Time: {run['start_time']}")
            if 'metrics.train_loss' in run:
                print(f"  Train Loss: {run['metrics.train_loss']:.4f}")
            if 'params.learning_rate' in run:
                print(f"  Learning Rate: {run['params.learning_rate']}")
            print()
    else:
        print("No runs found in this experiment.")
else:
    print("LoRA tuning experiment not found.")

## Step 6: Save Evaluation Results

Let's save our evaluation results for future reference and comparison.

In [ ]:
# Create evaluation results dictionary
eval_results = {
    "perplexity": {},
    "qualitative_examples": []
}

# Store perplexity results
for i, text in enumerate(test_texts):
    base_ppl = calculate_perplexity(text, base_model, tokenizer)
    fine_tuned_ppl = calculate_perplexity(text, model, tokenizer)
    eval_results["perplexity"][f"text_{i+1}"] = {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "base_model": base_ppl,
        "fine_tuned_model": fine_tuned_ppl,
        "improvement": base_ppl - fine_tuned_ppl
    }

# Store qualitative examples
for i, prompt in enumerate(test_prompts):
    base_response = generate_response(prompt, base_model, tokenizer)
        fine_tuned_response = generate_response(prompt, model, tokenizer)
    eval_results["qualitative_examples"].append({
        "prompt": prompt,
        "base_model_response": base_response,
        "fine_tuned_model_response": fine_tuned_response
    })

# Save to file
with open('./evaluation_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print("Evaluation results saved to './evaluation_results.json'")
print("\nSummary of Perplexity Improvements:")
for text_key, results in eval_results["perplexity"].items():
    print(f"{text_key}: {results['improvement']:.2f} points improvement")

## Summary

In this exercise, you learned how to:
1. Calculate perplexity to quantitatively evaluate language models
2. Perform qualitative evaluation by comparing model outputs
3. Use MLflow to track and compare experiments
4. Save evaluation results for future reference

## Key Takeaways

- Perplexity measures how well a model predicts a sequence (lower is better)
- Qualitative evaluation helps understand practical improvements in model behavior
- MLflow enables systematic tracking of experiments and model versions
- Evaluation is crucial for understanding the impact of fine-tuning

---
*Note: In a real-world scenario, you would evaluate on a larger validation set and use additional metrics like BLEU, ROUGE, or human evaluation for more comprehensive assessment.*